In [ ]:
# # Install dependencies for Unsloth + GPT-OSS
# !pip install --upgrade -qqq uv
# !uv pip install -qqq \
#     "torch>=2.8.0" "triton>=3.4.0" numpy pillow torchvision bitsandbytes "transformers==4.56.2" \
#     "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#     "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#     git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# # Install openai_harmony (Harmony protocol tools)
# !pip install -q openai-harmony jupyter_client pandas datasets


In [39]:
class CONFIG:
    TRAIN_SIZE = 10_000
    BATCH_SIZE = 16 
    EPOCHS = 1 
    LEARNING_RATE = 2e-4
    MAX_SEQ_LENGTH = 4096*2
    MODEL_PATH = None 
    KAGGLE=True 
    MASK_THINK = True
    KEEP_UNIQUE = False
    

cfg = CONFIG()
cfg.MODEL_PATH = "/kaggle/input/gpt-oss-20b-bnb-4bit/transformers/unsloth/1" if cfg.KAGGLE else "unsloth/gpt-oss-20b"

In [6]:
!python --version

Python 3.12.12


In [7]:
print("STARTING THE AHHHHHHHHHHHHHHHHHHHHHHHHHHHH")

STARTING THE AHHHHHHHHHHHHHHHHHHHHHHHHHHHH


In [8]:
if cfg.KAGGLE:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-py-3-12/unsloth' 'unsloth'


Using Python 3.12.12 environment at: /usr
Resolved 86 packages in 531ms                                        
Prepared 29 packages in 34.17s                                           
Uninstalled 21 packages in 2.04s
Installed 29 packages in 183ms                              
 + bitsandbytes==0.49.1
 + cut-cross-entropy==25.1.1
 - datasets==4.4.2
 + datasets==4.3.0
 - fsspec==2025.10.0
 + fsspec==2025.9.0
 + msgspec==0.20.0
 - multiprocess==0.70.18
 + multiprocess==0.70.16
 - nvidia-cublas-cu12==12.6.4.1
 + nvidia-cublas-cu12==12.8.4.1
 - nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-cupti-cu12==12.8.90
 - nvidia-cuda-nvrtc-cu12==12.6.77
 + nvidia-cuda-nvrtc-cu12==12.8.93
 - nvidia-cuda-runtime-cu12==12.6.77
 + nvidia-cuda-runtime-cu12==12.8.90
 - nvidia-cufft-cu12==11.3.0.4
 + nvidia-cufft-cu12==11.3.3.83
 - nvidia-cufile-cu12==1.11.1.6
 + nvidia-cufile-cu12==1.13.1.3
 - nvidia-curand-cu12==10.3.7.77
 + nvidia-curand-cu12==10.3.9.90
 - nvidia-cusolver-cu12==11.7.1.2
 + nvidia-cuso

In [9]:
try:
    from unsloth import FastLanguageModel
except:
    !uv pip install --system --no-index --find-links='//kaggle/input/unsloth-library/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

    

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-01-22 07:46:26.783827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769067987.192384     106 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769067987.329441     106 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769067988.419192     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769067988.419223     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769067988.419226     106 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [11]:
local_files_only = True if cfg.KAGGLE else False

In [ ]:
# del model
# del tokenizer

In [12]:
import torch
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.MODEL_PATH ,
    dtype = dtype, # None for auto detection
    max_seq_length = cfg.MAX_SEQ_LENGTH, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    local_files_only = local_files_only
)

==((====))==  Unsloth 2026.1.3: Fast Gpt_Oss patching. Transformers: 4.57.1.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.437 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
import polars as pl

files = [
    "/kaggle/input/nemotron-high-reasoning-pass-1-and-2-and-3-and-4/High_low_pass.jsonl",
    "/kaggle/input/nemotron-high-reasoning-pass-1-and-2-and-3-and-4/High_medium_pass.jsonl"
]

def load_all_columns_lazy(file_path):
    # 1. infer_schema_length=None forces it to scan ALL rows to find sparse columns like 'tools'
    lazy_df = pl.scan_ndjson(file_path, infer_schema_length=None, ignore_errors=True)
    
    # 2. "Reverse DropNA" Logic
    # We perform the filter HERE (lazily) instead of after loading.
    # This gives you the same result as "filtering later" but saves the RAM 
    # that would have been wasted loading the 'tools' rows.
    schema_keys = lazy_df.collect_schema().names()
    
    if "tools" in schema_keys:
        lazy_df = lazy_df.filter(pl.col("tools").is_null())
    
    # 3. REMOVED .select() -> Now returns ALL columns
    return lazy_df

# Setup scans
q1 = load_all_columns_lazy(files[0])
q2 = load_all_columns_lazy(files[1])

# Collect
print("Scanning and filtering (keeping all columns)...")

# CRITICAL: how="diagonal" ensures that if 'metadata' or 'url' is missing 
# in one file but present in the other, it doesn't crash.
df = pl.concat([q1, q2], how="diagonal").collect()

print(f"Loaded {len(df)} rows with columns: {df.columns}")
print(df.head())

Scanning and filtering (keeping all columns)...
Loaded 466396 rows with columns: ['expected_answer', 'problem', 'original_expected_answer', 'changed_answer_to_majority', 'data_source', 'messages', 'tools', 'used_in', 'metadata', 'license', 'url', 'user_url', 'user_name']
shape: (5, 13)
┌─────────────┬────────────┬────────────┬────────────┬───┬───────────┬──────┬──────────┬───────────┐
│ expected_an ┆ problem    ┆ original_e ┆ changed_an ┆ … ┆ license   ┆ url  ┆ user_url ┆ user_name │
│ swer        ┆ ---        ┆ xpected_an ┆ swer_to_ma ┆   ┆ ---       ┆ ---  ┆ ---      ┆ ---       │
│ ---         ┆ str        ┆ swer       ┆ jority     ┆   ┆ str       ┆ str  ┆ str      ┆ str       │
│ str         ┆            ┆ ---        ┆ ---        ┆   ┆           ┆      ┆          ┆           │
│             ┆            ┆ str        ┆ bool       ┆   ┆           ┆      ┆          ┆           │
╞═════════════╪════════════╪════════════╪════════════╪═══╪═══════════╪══════╪══════════╪═══════════╡
│ \( 3

In [ ]:
import polars as pl
import re

# --- CONFIGURATION ---
KEEP_INT_ONLY = True
CHANGE_SYSTEM_PROMPT = False 

# UPDATED: Hybrid Prompt
# 1. Sets the Math Olympiad Persona
# 2. Keeps the critical "Valid channels" instruction so format doesn't break
# 3. REMOVED the "tools" instruction because we filtered tools out (avoids hallucinations)
NEW_SYSTEM_PROMPT = (
    "You are an expert Math Olympiad solver. Your goal is to solve complex "
    "mathematical problems with rigorous, step-by-step reasoning.\n"
    "Knowledge cutoff: 2026-01\n"
    "Current date: 2026-01-22\n\n"
    "Reasoning: medium\n\n"
    "# Valid channels: analysis, final. Channel must be included for every message."
)

# --- 1. FILTERING ---
dataset = df.filter(
    pl.col("tools").is_null()
).drop(
    ["uuid", "original_expected_answer", "license", "used_in", "user_name", "user_url", "url", "tools"], 
    strict=False
)

# Integer Filter
if KEEP_INT_ONLY:
    dataset = dataset.with_columns(
        pl.col("expected_answer")
        .str.extract(r"(-?\d+\.?\d*)", 1)
        .cast(pl.Float64, strict=False)
        .alias("numeric_value")
    ).filter(
        pl.col("numeric_value").is_not_null()
    )

# Pre-sampling
PRE_SAMPLE_SIZE = int(cfg.TRAIN_SIZE * 1.5)
if len(dataset) > PRE_SAMPLE_SIZE:
    dataset = dataset.sample(n=PRE_SAMPLE_SIZE, seed=42, shuffle=True)

# --- 2. CLEANUP ---
def clean_messages(messages):
    if messages is None:
        return []
    
    new_history = []
    
    for msg in messages:
        new_msg = dict(msg)
        
        # Skip Tool Roles
        if new_msg.get('role') == 'tool':
            continue
            
        # Clean artifacts
        new_msg.pop('tool_calls', None)
        new_msg.pop('tool_call_id', None)

        if new_msg.get('reasoning_content'):
            new_msg['thinking'] = new_msg.pop('reasoning_content')
            
        new_msg = {k: v for k, v in new_msg.items() if v is not None}
        new_history.append(new_msg)
        
    return new_history

print("Cleaning messages...")
dataset = dataset.with_columns(
    pl.col("messages").map_elements(clean_messages, return_dtype=pl.Object).alias("AA")
)

# --- 3. TOKENIZE & SWAP ---
def apply_template_and_swap(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        reasoning_effort="medium"
    )
    
    if CHANGE_SYSTEM_PROMPT:
        # Replaces the default system message with our Hybrid one
        pattern = r"(<\|start\|>system<\|message\|>)(.*?)(<\|end\|>)"
        
        if re.search(pattern, text, flags=re.DOTALL):
            text = re.sub(
                pattern, 
                f"\\1{NEW_SYSTEM_PROMPT}\\3", 
                text, 
                count=1, 
                flags=re.DOTALL
            )
    
    return text

print(f"Tokenizing (System Prompt Swap: {CHANGE_SYSTEM_PROMPT})...")
dataset = dataset.with_columns(
    pl.col("AA").map_elements(apply_template_and_swap, return_dtype=pl.String).alias("text")
)

# --- 4. VERIFY ---
print("\n--- Final Prompt Check (First 500 chars) ---")
print(dataset["text"][0][:500])

if "developer" in dataset["text"][0][:500]:
    print("⚠️ WARNING: 'developer' role still found.")
else:
    print("✅ 'developer' role gone. System prompt is clean.")

In [50]:
# Show answers that failed the strict number cast
dropped_rows = df.filter(
    pl.col("tools").is_null()
).filter(
    pl.col("expected_answer").cast(pl.Float64, strict=False).is_null()
).select(["expected_answer"]).head(20)

print("--- REJECTED ANSWERS (Sample) ---")
print(dropped_rows)

--- REJECTED ANSWERS (Sample) ---
shape: (20, 1)
┌─────────────────────────────────┐
│ expected_answer                 │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ \( 3^{13} - 3 \)                │
│ \;y\bigl(\sqrt{x^{2}+y^{2}}+x\… │
│ 30\;\text{by}\;27               │
│ \(\frac{2abc}{ab + bc + ca}\)   │
│ a = b                           │
│ …                               │
│ a=8                             │
│ \( a = 1.465, b = \frac{\pi}{6… │
│ \[                              │
│ x = \frac{S^2 + b^2 + c^2 -…    │
│ y=x+1                           │
│ $(-\infty ;4]\cup \left[\frac{… │
└─────────────────────────────────┘


In [53]:
# dataset["text"][0]

'<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2026-01-22\n\nReasoning: medium\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: \'functions\'.<|end|><|start|>user<|message|>Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.\n\nFind a closed form for the integral \\( I = \\int_0^1 \\frac{(x^3 - 3x^2 + x) \\log(x - x^2)}{(x^2 - x + 1)^3} \\, dx \\).<|end|><|start|>assistant<|channel|>analysis<|message|>We need to evaluate I = ∫_0^1 ((x^3 - 3x^2 + x) * log(x - x^2)) / (x^2 - x + 1)^3 dx.\n\nSimplify function: x - x^2 = x(1 - x). So log(x - x^2) = log(x) + log(1 - x). The integrand is symmetric w.r.t. x -> 1-x? Let\'s check.\n\nDefine f(x) = (x^3 - 3 x^2 + x) / (x^2 - x + 1)^3. This appears odd under x -> 1 - x perhaps? Let\'s transform:\n\nLet t = 1 - x. Then x =

In [ ]:
# from datasets import Dataset

# # 1. Pre-processing
# train_data = df.copy()
# train_data.drop(columns=["uuid", "original_expected_answer", "license", "used_in", "user_name", "user_url", "url"], inplace=True, axis=1)

# # Filter out rows with tools (as per your snippet)
# train_data = train_data[train_data["tools"].isna()]
# train_data.drop(columns=["tools"], inplace=True, axis=1)

# def remove_none_keys(messages):
#     return [{k: v for k, v in entry.items() if v is not None} for entry in messages]

# def format_for_gpt_oss(example):
#     messages = example['messages']
    
#     # --- MODIFICATION START ---
#     # Initialize with the System Prompt
#     new_messages = [{
#         "role": "system", 
#         "content": "YOU ARE A MATH EXPERT"
#     }]
#     # --- MODIFICATION END ---
    
#     for msg in messages:
#         # Optional: Skip existing system prompts to strictly enforce your new one
#         if msg.get('role') == 'system':
#             continue

#         new_msg = msg.copy()
        
#         # 1. Rename 'reasoning_content' to 'thinking'
#         if 'reasoning_content' in new_msg:
#             new_msg['thinking'] = new_msg.pop('reasoning_content')
        
#         # 2. Ensure intermediate tool steps don't conflict
#         if new_msg.get('tool_calls') and new_msg.get('thinking'):
#              new_msg['content'] = "" 

#         new_messages.append(new_msg)
    
#     return {'messages': new_messages}

# # Apply to your dataset
# train_data_formatted = train_data.apply(format_for_gpt_oss, axis=1)
# train_data["messages"] = train_data_formatted

# # Extract the list of messages
# train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])

# # Create the final dataset slice
# dataset = train_data.iloc[0:cfg.TRAIN_SIZE].copy()
# dataset["AA"] = dataset["AA"].apply(remove_none_keys)

# # Apply Chat Template
# dataset["text"] = dataset.apply(lambda row: tokenizer.apply_chat_template(
#     row["AA"], 
#     tokenize=False, 
#     add_generation_prompt=True,
#     reasoning_effort="low"
# ), axis=1)

In [32]:
# Add LoRA adapters with rank 16
from datasets import Dataset

# hf_dataset = Dataset.from_pandas(dataset)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [38]:
import gc
import torch
gc.collect()


519

In [42]:
import pyarrow as pa
from datasets import Dataset

print("🔄 Converting Polars DataFrame to Hugging Face Dataset...")

# 1. Select only the columns needed for training to save RAM
# SFTTrainer strictly needs 'text'. We keep 'expected_answer' just for reference/debug if needed.
# If your dataset is huge, you can just do .select(["text"])
hf_arrow_table = dataset.select(["text"]).to_arrow()

# 2. Create the Hugging Face Dataset directly from Arrow (Zero-Copy)
hf_dataset = Dataset(hf_arrow_table)

# 3. Validation
print(f"✅ Hugging Face Dataset created successfully!")
print(f"Shape: {hf_dataset.shape}")
print(f"Column Names: {hf_dataset.column_names}")

# Verify the text field is correct for SFTTrainer
print("\n--- Sample Training Entry ---")
print(hf_dataset[0]["text"][:500]) # Print first 500 chars

🔄 Converting Polars DataFrame to Hugging Face Dataset...
✅ Hugging Face Dataset created successfully!
Shape: (1000, 1)
Column Names: ['text']

--- Sample Training Entry ---
<|start|>system<|message|>You are an expert Math Olympiad solver.
Knowledge cutoff: 2024-06
Current date: 2026-01-22

Reasoning: high

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are an expert Math Olympiad solver. Your goal is to solve complex mathematical problems with rigorous, step-by-step reasoning.

Reasoning: medium<|end


In [43]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=cfg.BATCH_SIZE,
        gradient_accumulation_steps=2,
        warmup_steps=5,
        num_train_epochs=1, 
       # max_steps=10 ,
        learning_rate=cfg.LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=666,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        group_by_length=True
    ),
)



Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [44]:
# CHANGE THIS: Point to the analysis channel instead of the final channel sometime
if not cfg.MASK_THINK:
    try:
        gpt_oss_kwargs = dict(
            instruction_part = "<|start|>user<|message|>", 
            response_part = "<|start|>assistant<|channel|>analysis<|message|>"
        )
        
        trainer = train_on_responses_only(
            trainer,
            **gpt_oss_kwargs,
        )
        TRAIN=True
    except:
        TRAIN = False
else:
    gpt_oss_kwargs = dict(
    instruction_part="<|start|>user<|message|>", 
    response_part="<|start|>assistant<|channel|>final<|message|>"
)
    trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)
    TRAIN = True



    


print(f"TRAINING READY AS NOT MASKING THE THINKING FROM TRAIN LOSS {cfg.MASK_THINK}")

Map (num_proc=30):   0%|          | 0/1000 [00:00<?, ? examples/s]

TRAINING READY AS NOT MASKING THE THINKING FROM TRAIN LOSS True


In [45]:
# 1. Train the model
if TRAIN:
    trainer_stats = trainer.train()

# 2. Memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# 3. Time stats
# 'train_runtime' is in seconds
train_time_seconds = trainer_stats.metrics.get('train_runtime', 0)
train_time_minutes = round(train_time_seconds / 60, 2)

print(f"Peak reserved memory = {used_memory} GB")
print(f"Total training time  = {train_time_minutes} minutes ({train_time_seconds:.2f} seconds)")

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 32
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 15,925,248 of 20,930,682,432 (0.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.797700
2,0.345700
3,0.283000
4,0.591300
5,0.648300
6,0.528700
7,0.499400
8,0.748400
9,0.424500
10,0.627600


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("gpt_oss_20b_nemotronv2_HIGH_FIXED")
tokenizer.save_pretrained("gpt_oss_20b_nemotronv2_HIGH_FIXED")
print("Model saved to 'gpt_oss_20b_nemotronv2_low'")